# LegalMitra — In-House Legal LLM Fine-Tuning Notebook
### Phase 66: 4-Bit QLoRA Fine-Tuning of Llama-3.1-8B-Instruct on Kaggle

This notebook trains a customized **LegalMitra-LLM** directly on Kaggle's free GPU (Tesla T4 or P100, 16GB VRAM) in approximately 1.5 to 2.5 hours.

**Core Objectives:**
1. Teach Llama-3.1 natural understanding of Indian legal inquiries, statutory citations, and IRAC arguments.
2. Ground outputs strictly in provided legal evidence to prevent hallucination.
3. Enforce adversarial defenses (defeating prompt injections, ungrounded statutes, and synthetic scaffold contamination).

---

## 1. Install Unsloth & Efficient Fine-Tuning Dependencies

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes datasets triton
!pip install rouge-score

## 2. Load Base Model in 4-Bit Quantization (Fits 16GB GPU VRAM)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection (Float16 on T4, Bfloat16 on Ampere)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Base model loaded successfully into 4-bit memory!")

## 3. Attach LoRA Parameter Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16, # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapter attached. Only ~0.5% of parameters will be trained!")

## 4. Load & Format LegalMitra Training Dataset

In [ ]:
from datasets import load_dataset

# Upload legal_instructions_train.jsonl to Kaggle dataset or load directly:
dataset = load_dataset("json", data_files={"train": "legal_instructions_train.jsonl", "val": "legal_instructions_val.jsonl"})

alpaca_prompt = """Below is an Indian legal query with governing statutory and precedent context. Write a grounded, well-reasoned legal explanation. Always cite authentic enactments and sections. If evidence is lacking or fictional, state so clearly.

### Legal Query:
{}

### Legal Context:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts = examples["context"]
    responses = examples["response"]
    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        text = alpaca_prompt.format(instruction, context, response) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Training set ready: {len(dataset['train'])} examples!")

## 5. Configure SFTTrainer & Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        max_steps=250, # ~2 epochs on batch
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="legalmitra_lora_checkpoints",
    ),
)

trainer_stats = trainer.train()
print("Training Complete! Final Stats:", trainer_stats)

## 6. Test Model Inference on an Unseen Query

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = alpaca_prompt.format(
    "Under Section 32A of the Insolvency and Bankruptcy Code, does the approval of a resolution plan immunize the corporate debtor from past offences?",
    "Statutory Authority: Insolvency and Bankruptcy Code, 2016\nSection 32A: Liability for prior offences shall cease and the corporate debtor shall not be prosecuted...",
    ""
)

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## 7. Export LoRA Adapter & Merged GGUF Weights

In [ ]:
# 1. Save standard LoRA adapters (lightweight ~150MB, fast to download)
model.save_pretrained("legalmitra_llama3_lora")
tokenizer.save_pretrained("legalmitra_llama3_lora")
print("Saved LoRA weights to 'legalmitra_llama3_lora'!")

# 2. Optional: Save 4-bit GGUF for local Ollama / llama.cpp serving
# model.save_pretrained_gguf("legalmitra_model_q4_k_m", tokenizer, quantization_method="q4_k_m")